# PanNuke Training on Kaggle - Simplified
Clone repo → Install deps → Train directly from raw data

## 1. Clone Repository

In [ ]:
import os
import subprocess

repo_path = '/kaggle/working/pannuke-segmentation'
if not os.path.exists(repo_path):
    !git clone https://github.com/phihungcr1701/Segmentation.git {repo_path}
    print(f"✅ Repository cloned to {repo_path}")
else:
    print(f"✅ Repository already exists at {repo_path}")

os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations tqdm numpy pillow matplotlib plotly streamlit

import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Verify Kaggle Dataset

In [ ]:
import os

dataset_path = '/kaggle/input/datasets/phihungcrr1701/pannuke'

print(f"Dataset path: {dataset_path}")
if os.path.exists(dataset_path):
    print(f"✅ Dataset found!\n")
    items = os.listdir(dataset_path)
    for item in sorted(items):
        full_path = os.path.join(dataset_path, item)
        if os.path.isdir(full_path):
            print(f"  📁 {item}/")
            sub_items = os.listdir(full_path)[:3]
            for sub in sub_items:
                print(f"     └─ {sub}")
        else:
            print(f"  📄 {item}")
else:
    print(f"❌ Dataset not found at {dataset_path}")

## 4. Configure Training Parameters

In [ ]:
import json

TRAINING_CONFIG = {
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'model': 'UNet',  # UNet, UNet++, ResNetUNet, ResNetUNet_pt, DeepLabV3+
    'batch_size': 16,
    'num_epochs': 50,
    'learning_rate': 1e-4,
    'loss_function': 'FocalDiceLoss',
    'train_fold': 1,
    'val_fold': 2,
    'root_dir': '/kaggle/input/datasets/phihungcrr1701/pannuke',  # Direct Kaggle dataset path
}

print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)
for key, value in TRAINING_CONFIG.items():
    print(f"{key:20} : {value}")
print("="*60)

config_path = os.path.join(repo_path, 'training_config.json')
with open(config_path, 'w') as f:
    json.dump(TRAINING_CONFIG, f, indent=2)
print(f"\n✅ Configuration saved to {config_path}")

## 5. Train the Model

In [ ]:
import subprocess

train_cmd = [
    'python', 'train.py',
    '--device', TRAINING_CONFIG['device'],
    '--model', TRAINING_CONFIG['model'],
    '--batch_size', str(TRAINING_CONFIG['batch_size']),
    '--num_epochs', str(TRAINING_CONFIG['num_epochs']),
    '--learning_rate', str(TRAINING_CONFIG['learning_rate']),
    '--loss', TRAINING_CONFIG['loss_function'],
    '--train_fold', str(TRAINING_CONFIG['train_fold']),
    '--val_fold', str(TRAINING_CONFIG['val_fold']),
    '--root_dir', TRAINING_CONFIG['root_dir'],
]

print(f"Starting training...\n")
print(f"Command: {' '.join(train_cmd)}\n")
print("="*60)

os.chdir(repo_path)
result = subprocess.run(train_cmd, capture_output=False)

print("\n" + "="*60)
if result.returncode == 0:
    print("✅ Training completed successfully!")
else:
    print("❌ Training failed!")
print("="*60)

## 6. Verify Checkpoints

In [ ]:
checkpoint_dir = os.path.join(repo_path, 'checkpoints', TRAINING_CONFIG['model'])

print(f"Checkpoint directory: {checkpoint_dir}\n")

if os.path.exists(checkpoint_dir):
    checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')])
    print(f"✅ Found {len(checkpoints)} checkpoint files:\n")
    
    for ckpt in checkpoints[-3:]:  # Show last 3
        ckpt_path = os.path.join(checkpoint_dir, ckpt)
        size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
        print(f"  {ckpt:30} ({size_mb:.2f} MB)")
    
    if len(checkpoints) > 3:
        print(f"  ... and {len(checkpoints) - 3} more")
    
    print(f"\n✅ Latest checkpoint: {checkpoints[-1]}")
else:
    print(f"❌ Checkpoint directory not found")

## 7. Save Results for Download

In [ ]:
import shutil
from datetime import datetime

output_dir = '/kaggle/working/outputs'
os.makedirs(output_dir, exist_ok=True)

print(f"Preparing outputs for download...\n")

# Copy checkpoints
checkpoint_src = os.path.join(repo_path, 'checkpoints')
checkpoint_dst = os.path.join(output_dir, 'checkpoints')

if os.path.exists(checkpoint_src):
    print(f"Copying checkpoints...")
    if os.path.exists(checkpoint_dst):
        shutil.rmtree(checkpoint_dst)
    shutil.copytree(checkpoint_src, checkpoint_dst)
    print(f"  ✅ Copied\n")

# Copy results
results_src = os.path.join(repo_path, 'results')
results_dst = os.path.join(output_dir, 'results')

if os.path.exists(results_src):
    print(f"Copying results...")
    if os.path.exists(results_dst):
        shutil.rmtree(results_dst)
    shutil.copytree(results_src, results_dst)
    print(f"  ✅ Copied\n")

# Copy training config
if os.path.exists(config_path):
    shutil.copy(config_path, os.path.join(output_dir, 'training_config.json'))
    print(f"Copying config...")
    print(f"  ✅ Copied\n")

print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model           : {TRAINING_CONFIG['model']}")
print(f"Loss Function   : {TRAINING_CONFIG['loss_function']}")
print(f"Batch Size      : {TRAINING_CONFIG['batch_size']}")
print(f"Epochs          : {TRAINING_CONFIG['num_epochs']}")
print(f"Learning Rate   : {TRAINING_CONFIG['learning_rate']}")
print(f"Timestamp       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)
print(f"\n✅ All outputs saved to: {output_dir}")
print(f"Ready to download!")